# PCA on Front-Colocated Properties: Global Visualization

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport matplotlib.colors as mcolorsimport matplotlib.cm as cmfrom matplotlib.patches import Rectangleimport xarray as xrimport pandas as pdfrom pathlib import Pathimport json, refrom scipy.stats import gaussian_kdeimport cartopy.crs as ccrsimport cartopy.feature as cfeaturefrom fronts.properties.colocation import colocate_fronts_with_propertiesplt.rcParams['figure.figsize'] = (16, 9)plt.rcParams['font.size'] = 10%matplotlib inlineprint('Imports OK')

## 1. Load Pre-Computed ResultsLoads the labeled-fronts array, geometric-property parquet, lat/lon coordinates, and **pre-computed colocation outputs** produced by `build_v1.py → group_fronts()` and `colocate_fronts()`.

In [ ]:
# ─── FILE PATHS ─── update these ───────────────────────────────────────────results_dir  = '/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/group_fronts/v1/'          # directory with parquet / npy / jsoncoords_file  = '/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/lohoff/group_fronts/LLC_coords_lat_lon.nc'ref_file     = '/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/derived/LLC4320_2012-11-09T12_00_00_gradb2_v1.nc'run_tag      = 'v1_bin_A'  # run identifier for file naming# ─────────────────────────────────────────────────────────────────────────────# Extract timestamp from ref_file namepattern = r'(\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2})'match   = re.search(pattern, ref_file)if not match:    raise ValueError(f'Cannot extract timestamp from: {ref_file}')time_str      = match.group(1).replace('_', ':')   # 2012-11-09T12:00:00time_str_safe = match.group(1).replace('-', '').replace(':', '')  # 20121109T12_00_00time_str_safe = re.sub(r'T(\d{2})(\d{2})(\d{2})', r'T__',                       time_str.replace('-','').replace(':',''), count=1)print(f'Timestep: {time_str}  (safe: {time_str_safe})')# Construct standard filenames with run_tagmetadata_file  = Path(results_dir) / f'metadata_{time_str_safe}_{run_tag}.json'labeled_file   = Path(results_dir) / f'labeled_fronts_global_{time_str_safe}_{run_tag}.npy'geometry_file  = Path(results_dir) / f'global_front_geometry_{time_str_safe}_{run_tag}.parquet'coloc_file     = Path(results_dir) / f'front_properties_{time_str_safe}_{run_tag}.parquet'# Load metadatawith open(metadata_file) as f:    metadata = json.load(f)downsample_factor = metadata.get('downsample_factor', None)print(f"Shape: {metadata['shape']},  Fronts: {metadata['num_fronts']:,}")if downsample_factor:    print(f'  ⚠️  Downsampled ×{downsample_factor}')# Load label maplabeled_global = np.load(labeled_file)print(f'labeled_global: {labeled_global.shape}')# Load geometric propertiesdf_global = pd.read_parquet(geometry_file)print(f'df_global: {len(df_global):,} fronts × {len(df_global.columns)} cols')# Load colocation (enriched properties)df_coloc = pd.read_parquet(coloc_file)print(f'df_coloc: {len(df_coloc):,} fronts × {len(df_coloc.columns)} cols')# Merge geometric + colocation data on front labeldf_enriched = df_global.merge(df_coloc, left_on='label', right_on='flabel', how='inner')print(f'df_enriched: {len(df_enriched):,} fronts × {len(df_enriched.columns)} cols')print(f'  Columns: {list(df_enriched.columns)[:10]}...')# Load coordinatesds_coords  = xr.open_dataset(coords_file)lat_global = ds_coords['lat'].values if 'lat' in ds_coords else ds_coords['YC'].valueslon_global = ds_coords['lon'].values if 'lon' in ds_coords else ds_coords['XC'].valuesds_coords.close()if downsample_factor:    lat_global = lat_global[::downsample_factor, ::downsample_factor]    lon_global = lon_global[::downsample_factor, ::downsample_factor]# Roll arrays so columns run -180 → +180sample_row  = lat_global.shape[0] // 2min_lon_col = int(np.argmin(lon_global[sample_row, :]))shift       = -min_lon_colif min_lon_col != 0:    print(f'Rolling by {shift} cols to align longitude axis')    lon_global    = np.roll(lon_global,    shift, axis=1)    lat_global    = np.roll(lat_global,    shift, axis=1)    labeled_global = np.roll(labeled_global, shift, axis=1)else:    print('No rolling needed')print(f'\n✓ Pre-computed results loaded.  Array shape: {labeled_global.shape}')

## 2. Load Physical Property MapsProperty maps are loaded from standard NetCDF files. Edit `property_names` to load only what you need for PCA.

In [ ]:
# ─── SETTINGS ────────────────────────────────────────────────────────────────properties_dir = '/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/derived'version        = '1'timestamp      = '2012-11-09T12_00_00'property_names = [    'gradb2',    'gradtheta2',    'gradeta2',    'gradsalt2',    'relative_vorticity',    'strain_mag',    'okubo_weiss',    'divergence',    'coriolis_f',    'frontogenesis_tendency',]# Files auto-resolved as: {properties_dir}/LLC4320_{timestamp}_{prop_name}_v{version}.nc# ─────────────────────────────────────────────────────────────────────────────property_files = {    name: (        str(Path(properties_dir) / f'LLC4320_{timestamp}_{name}_v{version}.nc'),        name,    )    for name in property_names}

In [ ]:
import xarray as xrfrom fronts.properties.colocation import _load_property_fileproperty_arrays = {}for prop_name, src in property_files.items():    arr = _load_property_file(src)          # handles .nc tuples, .npy, or arrays    arr = arr.squeeze()                      # drop any singleton time/depth dims    if downsample_factor:        arr = arr[::downsample_factor, ::downsample_factor]    if min_lon_col != 0:                     # same longitude roll as labeled array        arr = np.roll(arr, shift, axis=1)    property_arrays[prop_name] = arr    src_label = src[0] if isinstance(src, (list, tuple)) else (                    str(src) if not isinstance(src, np.ndarray) else '<array>')    print(f"  {prop_name:25s}  shape={arr.shape}  "          f"range=[{float(np.nanmin(arr)):.3g}, {float(np.nanmax(arr)):.3g}]")print(f'\n✓ {len(property_arrays)} property arrays loaded and aligned')

## 3. Spatial Binning MapsAggregate the per-front co-located property statistics into regular lat/lon bins (~100 grid-pixel bins by default).* **Panel (a)** — mean of the co-located property stat across all fronts in the bin  * **Panel (b)** — standard deviation of that same per-front stat across fronts in the bin> This shows *where* the property is systematically elevated/suppressed at fronts, and *where* there is large front-to-front variability.

In [ ]:
from scipy.stats import binned_statistic_2d# ─── SETTINGS ────────────────────────────────────────────────────────────────# Column in df_enriched to bin.  Typical choices:#   'relative_vorticity_mean', 'rossby_number_mean', 'relative_vorticity_std'BIN_PROP     = 'okubo_weiss_median'  # Bin size expressed as a fraction of labeled_global pixels.# 100 => bins that are ~100 native-grid-pixels wide.BIN_SIZE_PIX = 100BIN_CMAP_A   = 'RdBu_r'   # colormap for mean panel (diverging)BIN_CMAP_B   = 'viridis'   # colormap for std  panel (sequential)BIN_PCT      = 95           # symmetric percentile clip for colormapsMIN_FRONTS   = 2            # minimum fronts per bin to show (mask singletons)# ─────────────────────────────────────────────────────────────────────────────if BIN_PROP not in df_enriched.columns:    raise ValueError(f"'{BIN_PROP}' not found in df_enriched. "                     f"Available cols: {[c for c in df_enriched.columns if '_mean' in c or '_std' in c]}")# Number of bins derived from labeled_global shape and requested pixel sizen_lat_bins = max(5, labeled_global.shape[0] // BIN_SIZE_PIX)n_lon_bins = max(5, labeled_global.shape[1] // BIN_SIZE_PIX)# Bin edges in geographic coordinateslat_min_g, lat_max_g = float(lat_global.min()), float(lat_global.max())lon_min_g, lon_max_g = float(lon_global.min()), float(lon_global.max())lat_edges = np.linspace(lat_min_g, lat_max_g, n_lat_bins + 1)lon_edges = np.linspace(lon_min_g, lon_max_g, n_lon_bins + 1)lat_ctr   = 0.5 * (lat_edges[:-1] + lat_edges[1:])lon_ctr   = 0.5 * (lon_edges[:-1] + lon_edges[1:])# Work with rows that have valid coordinates AND valid property valuesdf_b = df_enriched[['centroid_lat', 'centroid_lon', BIN_PROP]].dropna()print(f"Binning '{BIN_PROP}'  into {n_lat_bins} × {n_lon_bins} bins")print(f"  Lat range: [{lat_min_g:.1f}°, {lat_max_g:.1f}°]")print(f"  Lon range: [{lon_min_g:.1f}°, {lon_max_g:.1f}°]")print(f"  Fronts with valid data: {len(df_b):,}")# binned_statistic_2d: first dimension = lat (rows), second = lon (columns)kw = dict(bins=[lat_edges, lon_edges])mean_stat, _, _, _ = binned_statistic_2d(    df_b['centroid_lat'].values, df_b['centroid_lon'].values,    df_b[BIN_PROP].values, statistic='mean', **kw)std_stat,  _, _, _ = binned_statistic_2d(    df_b['centroid_lat'].values, df_b['centroid_lon'].values,    df_b[BIN_PROP].values, statistic='std',  **kw)cnt_stat,  _, _, _ = binned_statistic_2d(    df_b['centroid_lat'].values, df_b['centroid_lon'].values,    df_b[BIN_PROP].values, statistic='count', **kw)# Mask bins with too few frontssparse = cnt_stat < MIN_FRONTSmean_stat = np.where(sparse, np.nan, mean_stat)std_stat  = np.where(sparse, np.nan, std_stat)# Colormap limitsfm = mean_stat[np.isfinite(mean_stat)]fs = std_stat[np.isfinite(std_stat)]clip_a  = float(np.percentile(np.abs(fm), BIN_PCT)) if len(fm) else 1.0vmin_a, vmax_a = -clip_a, clip_avmin_b  = 0.0vmax_b  = float(np.percentile(fs, BIN_PCT)) if len(fs) else 1.0# ── Plot ──────────────────────────────────────────────────────────────────────prop_label = BIN_PROP.replace('_', ' ').title()# meshgrid of BIN EDGES for pcolormesh (gives exact bin boundaries)lon_e2d, lat_e2d = np.meshgrid(lon_edges, lat_edges)HAS_CARTOPY = True  # set to False if cartopy not availabletfm = ccrs.PlateCarree() if HAS_CARTOPY else Noneif HAS_CARTOPY:    fig, axes = plt.subplots(1, 2, figsize=(22, 7),                             subplot_kw={'projection': ccrs.PlateCarree()})    pcm_kw = dict(transform=tfm, rasterized=True)else:    fig, axes = plt.subplots(1, 2, figsize=(22, 7))    pcm_kw = dict(rasterized=True)panels = [    ('(a)  Mean',  mean_stat, BIN_CMAP_A, vmin_a, vmax_a),    ('(b)  Std',   std_stat,  BIN_CMAP_B, vmin_b, vmax_b),]for ax, (panel_title, data, cmap, vmin, vmax) in zip(axes, panels):    # pcolormesh with 2-D edge arrays for true bin boundaries    pm = ax.pcolormesh(lon_e2d, lat_e2d, np.ma.masked_invalid(data),                       cmap=cmap, vmin=vmin, vmax=vmax, **pcm_kw)    plt.colorbar(pm, ax=ax, shrink=0.75,                 label=prop_label, extend='both' if vmin < 0 else 'max')    if HAS_CARTOPY:        gl = ax.gridlines(crs=tfm, draw_labels=True,                          linewidth=0.4, color='gray', alpha=0.4, linestyle='--')        gl.top_labels   = False        gl.right_labels = False        ax.set_global()    else:        ax.set_xlabel('Longitude (°E)')        ax.set_ylabel('Latitude (°N)')        ax.grid(True, alpha=0.3)    ax.set_title(f'{panel_title}  |  {prop_label}', fontsize=22, fontweight='bold')bin_deg_lat = (lat_max_g - lat_min_g) / n_lat_binsbin_deg_lon = (lon_max_g - lon_min_g) / n_lon_binsfilled = int(np.isfinite(mean_stat).sum())plt.suptitle(    f'Front-co-located  {prop_label}  per lat/lon bin\n'    f'{n_lat_bins} × {n_lon_bins} bins  (≈{bin_deg_lat:.1f}° × {bin_deg_lon:.1f}° each)  |  '    f'{filled}/{n_lat_bins*n_lon_bins} bins occupied  |  {len(df_b):,} fronts',    fontsize=16, fontweight='bold', y=1.01)plt.tight_layout()plt.show()print(f'✓  Spatial binning map  ({filled} occupied bins, '      f'min {int(cnt_stat[cnt_stat > 0].min()) if (cnt_stat > 0).any() else 0} '      f'to max {int(cnt_stat.max())} fronts/bin)')

## 4. PCA on Front-Colocated PropertiesPrincipal Component Analysis across all co-located physical properties.  Each front is a point in property space; PCA finds the axes of maximum co-variability across the full front catalogue.**Plots produced**1. **Scree plot** — variance explained per PC; dashed line at 80% cumulative2. **Loadings bar chart** — which properties define each PC (red = positive, blue = negative)3. **PC score maps** — mean front PC score binned onto a global 2° lat/lon grid4. **PC1 vs PC2 biplot** — each front as a point, coloured by grad-b², with loading vectors overlaid

In [ ]:
from fronts.properties.pca import run_pca_fronts# ─── Derived / normalised properties (add before PCA_PROPS) ──────────────────df_enriched['norm_relative_vorticity_median']   = (df_enriched['relative_vorticity_median']                                        / df_enriched['coriolis_f_median'])df_enriched['abs_relative_vorticity_median']   = (df_enriched['relative_vorticity_median'].abs())#df_enriched['speed_median']     = np.sqrt(df_enriched['U_median']**2 + df_enriched['V_median']**2)df_enriched['norm_strain_median']     = (df_enriched['strain_mag_median']                                        / df_enriched['coriolis_f_median'].abs())# ─── SETTINGS ────────────────────────────────────────────────────────────────# Properties to include — must match keys loaded in Section 2.# coriolis_f is excluded: it encodes geography, not dynamics.PCA_PROPS = [    'gradb2_median',    'abs_relative_vorticity_median',    'strain_mag_median',    #'divergence_median',    'gradtheta2_median',    'gradsalt2_median',]# Filter to columns that actually exist in df_enrichedPCA_PROPS = [c for c in PCA_PROPS if c in df_enriched.columns]print(f'Running PCA on {len(PCA_PROPS)} properties: {PCA_PROPS}')N_PCA_COMPONENTS = None   # None → keep all; set e.g. 4 to retain fewer# Lat/lon bins for spatial mapsPCA_LAT_BINS = 90    # 2° binsPCA_LON_BINS = 180   # 2° bins# ─── RUN ─────────────────────────────────────────────────────────────────────pca_result = run_pca_fronts(    df_enriched, PCA_PROPS,    n_components=N_PCA_COMPONENTS,    standardize=True,)print()print(pca_result.summary().to_string(index=False))print()print('Top 3 loadings per PC:')print(pca_result.top_loadings(n=3).to_string(index=False))# Pre-compute spatial maps for all PCslat_pca = df_enriched.loc[pca_result.scores.index, 'centroid_lat']lon_pca = df_enriched.loc[pca_result.scores.index, 'centroid_lon']pca_maps = pca_result.to_map(lat_pca, lon_pca,                              lat_bins=PCA_LAT_BINS, lon_bins=PCA_LON_BINS)

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.gridspec as gridspecimport numpy as npdef _short(col):    """Strip _mean/_median/_std suffix and replace underscores with spaces."""    return re.sub(r'_(mean|median|std)$', '', col).replace('_', ' ')n_pcs   = len(pca_result.explained_variance_ratio)pc_lbls = [f'PC{i+1}' for i in range(n_pcs)]# ── Figure 1: Scree plot ──────────────────────────────────────────────────────fig1, ax = plt.subplots(figsize=(max(5, n_pcs * 0.7 + 1), 4))ax.bar(pc_lbls, pca_result.explained_variance_ratio * 100,       color='steelblue', edgecolor='white', linewidth=0.5)ax2 = ax.twinx()ax2.plot(pc_lbls, pca_result.cumulative_variance_ratio * 100,         color='tomato', marker='o', ms=5, lw=1.5, label='Cumulative')ax2.axhline(80, color='tomato', lw=0.8, ls='--', alpha=0.6)ax2.set_ylabel('Cumulative variance explained (%)', color='tomato')ax2.tick_params(axis='y', colors='tomato')ax2.set_ylim(0, 105)ax.set_xlabel('Principal Component')ax.set_ylabel('Variance explained (%)')ax.set_title('Scree Plot — PCA on Front-Colocated Properties')fig1.tight_layout()plt.show()# ── Figure 2: Loadings bar chart ──────────────────────────────────────────────n_show = min(n_pcs, 4)   # show at most 4 PCs side by sideshort_names = [p.replace('_mean', '').replace('_', '\n') for p in PCA_PROPS]fig2, axes = plt.subplots(1, n_show,                           figsize=(n_show * 3.2, max(3, len(PCA_PROPS) * 0.55 + 1)),                           sharey=True)if n_show == 1:    axes = [axes]vmax = pca_result.loadings.abs().values.max()for k, ax in enumerate(axes):    pc_col = f'PC{k+1}'    vals   = pca_result.loadings[pc_col].values    colors = ['#d7191c' if v > 0 else '#2c7bb6' for v in vals]    ax.barh(range(len(PCA_PROPS)), vals, color=colors, edgecolor='white', lw=0.4)    ax.axvline(0, color='k', lw=0.8)    ax.set_xlim(-vmax * 1.15, vmax * 1.15)    ax.set_title(        f'{pc_col}\n({pca_result.explained_variance_ratio[k]*100:.1f}%)',        fontsize=10)    ax.set_xlabel('Loading')    if k == 0:        ax.set_yticks(range(len(PCA_PROPS)))        ax.set_yticklabels(short_names, fontsize=8)fig2.suptitle('PCA Loadings — contribution of each property to each PC',              fontsize=11)fig2.tight_layout()plt.show()# ── Figure 3: PC score maps (up to 4 PCs, 2 columns) ────────────────────────n_map = min(n_pcs, 4)ncols = 2nrows = (n_map + 1) // 2fig3, axes = plt.subplots(nrows, ncols, figsize=(24, nrows * 8))axes = np.array(axes).reshape(-1) if n_map > 1 else [axes]for k in range(n_map):    ax  = axes[k]    pc_col = f'PC{k+1}'    grid, lat_e, lon_e = pca_maps[pc_col]    lon_e2d, lat_e2d   = np.meshgrid(lon_e, lat_e)    vabs = np.nanpercentile(np.abs(grid), 97)    pm = ax.pcolormesh(lon_e2d, lat_e2d, np.ma.masked_invalid(grid),                       cmap='RdBu_r', vmin=-vabs, vmax=vabs,                       shading='auto', rasterized=True)    plt.colorbar(pm, ax=ax, orientation='horizontal', pad=0.05, fraction=0.04,                 label='Mean PC score per bin')    ax.set_xlim(-180, 180)    ax.set_ylim(-90, 90)    ax.set_xlabel('Longitude')    ax.set_ylabel('Latitude')    evr = pca_result.explained_variance_ratio[k] * 100    # Top-2 loading labels for title    top2 = pca_result.loadings[pc_col].abs().nlargest(2).index.tolist()    top2_str = ', '.join(p.replace('_mean', '') for p in top2)    ax.set_title(f'{pc_col}  ({evr:.1f}% var)\n[{top2_str}]', fontsize=10)for k in range(n_map, len(axes)):   # hide unused panels    axes[k].set_visible(False)fig3.suptitle('Global Spatial Distribution of PC Scores\n(mean front PC score per 2° bin)',              fontsize=12)fig3.tight_layout()plt.show()# ── Figure 4: PC1 vs PC2 biplot ───────────────────────────────────────────────# Auto-detect the color column: try _median then _mean suffixBIPLOT_COLOR_BASE = 'gradb2'_candidates = [f'{BIPLOT_COLOR_BASE}_median', f'{BIPLOT_COLOR_BASE}_mean',               BIPLOT_COLOR_BASE]BIPLOT_COLOR = next((c for c in _candidates if c in df_enriched.columns), None)CLIP_PCT = 99   # percentile used to clip axes and scale arrowsif BIPLOT_COLOR is not None:    idx = pca_result.scores.index    gb2 = df_enriched.loc[idx, BIPLOT_COLOR].values    s1  = pca_result.scores['PC1'].values    s2  = pca_result.scores['PC2'].values    # Clip axes to CLIP_PCT percentile to prevent outliers from collapsing the plot    x_lo, x_hi = np.nanpercentile(s1, [100 - CLIP_PCT, CLIP_PCT])    y_lo, y_hi = np.nanpercentile(s2, [100 - CLIP_PCT, CLIP_PCT])    # Symmetric around 0 for a cleaner look    x_abs = max(abs(x_lo), abs(x_hi))    y_abs = max(abs(y_lo), abs(y_hi))    # Subsample for scatter legibility    N_BI = 30_000    if len(s1) > N_BI:        rng2 = np.random.default_rng(0)        sel  = rng2.choice(len(s1), N_BI, replace=False)        s1_p, s2_p, gb2_p = s1[sel], s2[sel], gb2[sel]    else:        s1_p, s2_p, gb2_p = s1, s2, gb2    vmax_c = np.nanpercentile(gb2_p, 98)    fig4, ax = plt.subplots(figsize=(7, 6))    sc = ax.scatter(s1_p, s2_p, c=gb2_p, cmap='magma_r',                    vmin=0, vmax=vmax_c,                    s=4, alpha=0.4, linewidths=0, rasterized=True)    plt.colorbar(sc, ax=ax, label=f'{_short(BIPLOT_COLOR)} (front strength)')    # Loading vectors — scale to the clipped axis range so arrows are visible    arrow_scale = min(x_abs, y_abs) * 0.85    arrow_colors = plt.cm.Set2(np.linspace(0, 1, len(PCA_PROPS)))    for i_p, prop in enumerate(PCA_PROPS):        lx = pca_result.loadings.loc[prop, 'PC1'] * arrow_scale        ly = pca_result.loadings.loc[prop, 'PC2'] * arrow_scale        clr = arrow_colors[i_p]        ax.annotate('', xy=(lx, ly), xytext=(0, 0),                    arrowprops=dict(arrowstyle='->', color=clr, lw=1.8))        ax.text(lx * 1.8, ly * 1.8, _short(prop),                fontsize=7.5, ha='center', va='center', color=clr,                bbox=dict(boxstyle='round,pad=0.1', fc='#1a1a2e', ec='none', alpha=0.7))    ax.set_xlim(-x_abs * 1.1, x_abs * 1.1)    ax.set_ylim(-y_abs * 1.1, y_abs * 1.1)    ax.axhline(0, color='gray', lw=0.6, ls='--')    ax.axvline(0, color='gray', lw=0.6, ls='--')    ax.set_xlabel(f"PC1  ({pca_result.explained_variance_ratio[0]*100:.1f}%)")    ax.set_ylabel(f"PC2  ({pca_result.explained_variance_ratio[1]*100:.1f}%)")    ax.set_facecolor('#1a1a2e')    fig4.patch.set_facecolor('#1a1a2e')    ax.tick_params(colors='white')    ax.xaxis.label.set_color('white')    ax.yaxis.label.set_color('white')    ax.spines[['bottom','left','top','right']].set_color('gray')    ax.set_title('PC1 vs PC2 — fronts in score space\ncoloured by front strength',                 color='white')    fig4.tight_layout()    plt.show()    print(f'Biplot: axes clipped to {CLIP_PCT}th pct  '          f'[PC1: {-x_abs:.2f}–{x_abs:.2f}, PC2: {-y_abs:.2f}–{y_abs:.2f}]')else:    print(f'Biplot skipped: no column matching {BIPLOT_COLOR_BASE!r} found in df_enriched')